# 04: Paris / Lyon / Marseille: résultats par secteur + prix au m²

Produces a single JSON per city: one row per electoral secteur with `winning_bloc` and `median_prix_m2`.  
This is the data layer for the PLM choropleth map viz.

**Sources:**
- `Paris_Lyon_BV.parquet`: BV-level T2 results for Paris, Lyon, Marseille
- `ValeursFoncieres-2024.txt`: DVF 2024 residential transactions (prix au m²)
- `nuances.csv`: nuance code → political bloc mapping

**Pipeline:**
1. Aggregate BV votes → secteur level → idxmax → winning nuance → bloc
2. DVF → filter PLM arrondissements → median prix/m² per arrondissement
3. Secteur → arrondissement mapping dict → join
4. Export `data/processed/plm-secteurs.json`

**Data gaps (noted, not filled):**
- Paris 7e, 13e, 16e: uncontested at T2: no BV data, excluded from election layer
- Lyon 5e, 6e: same, excluded from election layer
- Prix m² data still available for all arrondissements from DVF

## 1. Load BV parquet

In [ ]:
import pandas as pd

bv = pd.read_parquet('../data/raw/Paris_Lyon_BV.parquet')

print(f'Shape: {bv.shape}')
print('\nCities:')
print(bv.groupby(['Code département', 'Libellé département'])['Code secteur'].nunique().rename('secteurs'))
print('\nBVs per secteur (sample):')
print(bv.groupby('Code secteur').size().head(10))

**Finding:** 1,483 BVs across 29 secteurs, 8 Marseille, 7 Lyon, 14 Paris. `Voix 1/2` are always populated; `Voix 3/4` are NaN where fewer lists competed. We sum raw vote counts across BVs to get secteur-level totals, then idxmax to find the winner.

## 2. Secteur → arrondissement mapping

Derived from the liste names in the BV data (e.g. "1e ET 7e ARRONDISSEMENTS"). Maps each secteur code to the DVF `code_commune` values it covers.  
DVF codes: Paris = `75101–75120`, Lyon = `69381–69389`, Marseille = `13201–13216`.

In [ ]:
# Secteur code → list of DVF code_commune values
# Paris: SR01 = Paris Centre (1e-4e merged 2023); SR07/13/16 absent = uncontested T2
# Lyon: SR05/06 absent = uncontested T2
# Marseille: confirmed from liste labels (pairs of arrondissements)

SECTEUR_ARRDTS = {
    # Paris
    '75056SR01': ['75101', '75102', '75103', '75104'],  # Paris Centre
    '75056SR05': ['75105'],
    '75056SR06': ['75106'],
    '75056SR08': ['75108'],
    '75056SR09': ['75109'],
    '75056SR10': ['75110'],
    '75056SR11': ['75111'],
    '75056SR12': ['75112'],
    '75056SR14': ['75114'],
    '75056SR15': ['75115'],
    '75056SR17': ['75117'],
    '75056SR18': ['75118'],
    '75056SR19': ['75119'],
    '75056SR20': ['75120'],
    # Lyon (5e=69385, 6e=69386 absent from BV data)
    '69123SR01': ['69381'],
    '69123SR02': ['69382'],
    '69123SR03': ['69383'],
    '69123SR04': ['69384'],
    '69123SR07': ['69387'],
    '69123SR08': ['69388'],
    '69123SR09': ['69389'],
    # Marseille: confirmed from liste names
    '13055SR01': ['13201', '13207'],  # 1e + 7e
    '13055SR02': ['13202', '13203'],  # 2e + 3e
    '13055SR03': ['13204', '13205'],  # 4e + 5e
    '13055SR04': ['13206', '13208'],  # 6e + 8e
    '13055SR05': ['13209', '13210'],  # 9e + 10e
    '13055SR06': ['13211', '13212'],  # 11e + 12e
    '13055SR07': ['13213', '13214'],  # 13e + 14e
    '13055SR08': ['13215', '13216'],  # 15e + 16e
}

print(f'Secteurs in mapping: {len(SECTEUR_ARRDTS)}')
print(f'Secteurs in BV data: {bv["Code secteur"].nunique()}')

# Verify all BV secteurs are covered
missing = set(bv['Code secteur'].unique()) - set(SECTEUR_ARRDTS.keys())
print(f'BV secteurs not in mapping: {missing}')

## 3. Aggregate BV votes → secteur → winning bloc

In [ ]:
voix_cols = ['Voix 1', 'Voix 2', 'Voix 3', 'Voix 4']

# Sum raw vote counts + abstention/inscription counts from BV to secteur
agg_cols = voix_cols + ['Abstentions', 'Inscrits']
secteur_voix = (
    bv.groupby('Code secteur')[agg_cols]
    .sum(min_count=1)  # min_count=1: all-NaN column stays NaN (no lists ran)
    .reset_index()
)

# Abstention rate at secteur level: sum(Abstentions) / sum(Inscrits)
secteur_voix['abstention_rate'] = (
    secteur_voix['Abstentions'] / secteur_voix['Inscrits'] * 100
).round(1)

# idxmax across Voix columns → column name of the highest vote count
# fill NaN with -1 so they're never picked as max
winning_col = secteur_voix[voix_cols].fillna(-1).idxmax(axis=1)

# Extract list number from column name: 'Voix 3' → '3'
winning_num = winning_col.str.split(' ').str[-1]
secteur_voix['winning_list_num'] = winning_num

print(secteur_voix[['Code secteur', 'Voix 1', 'Voix 2', 'Voix 3', 'Voix 4', 'abstention_rate', 'winning_list_num']].to_string())

In [ ]:
# For each secteur, pull the Nuance liste X where X = winning_list_num
# One representative BV row per secteur is enough: nuance is the same across all BVs in a secteur
bv_first = bv.drop_duplicates('Code secteur').set_index('Code secteur')

def get_winning_nuance(row):
    num = row['winning_list_num']
    secteur = row['Code secteur']
    return bv_first.loc[secteur, f'Nuance liste {num}']

secteur_voix['winning_nuance'] = secteur_voix.apply(get_winning_nuance, axis=1)

# Also pull Libellé secteur and city label
secteur_voix['libelle_secteur'] = secteur_voix['Code secteur'].map(bv_first['Libellé secteur'])
secteur_voix['code_dept'] = secteur_voix['Code secteur'].map(bv_first['Code département'])
secteur_voix['ville'] = secteur_voix['code_dept'].map({'75': 'Paris', '69': 'Lyon', '13': 'Marseille'})

print(secteur_voix[['Code secteur', 'ville', 'libelle_secteur', 'winning_nuance']].to_string())

In [ ]:
# Join nuances.csv → winning_bloc
nuances = pd.read_csv('../data/raw/nuances.csv', sep=';')
nuances_liste = nuances[nuances['type_nuance'] == 'liste']

secteur_voix = secteur_voix.merge(
    nuances_liste[['libelle_nuance', 'bloc']].rename(columns={'bloc': 'winning_bloc'}),
    left_on='winning_nuance',
    right_on='libelle_nuance',
    how='left'
)

print('Unmatched blocs:', secteur_voix['winning_bloc'].isna().sum())
print()
print(secteur_voix[['ville', 'libelle_secteur', 'winning_nuance', 'winning_bloc']].to_string())

## 4. DVF: median prix/m² per arrondissement

Filtered to PLM arrondissement `code_commune` ranges:  
Paris `75101–75120`, Lyon `69381–69389`, Marseille `13201–13216`.  
Same cleaning logic as notebook 02: Vente + Appartement/Maison, cap at €20,000/m².

In [ ]:
cols = ['Nature mutation', 'Type local', 'Code departement', 'Code commune',
        'Valeur fonciere', 'Surface reelle bati']

dvf = pd.read_csv('../data/raw/ValeursFoncieres-2025.txt', sep='|',
                  usecols=cols, low_memory=False)

print(f'Loaded: {len(dvf):,} rows')

# Filter to residential sales only
dvf = dvf[
    (dvf['Nature mutation'] == 'Vente') &
    (dvf['Type local'].isin(['Appartement', 'Maison']))
].copy()

print(f'After residential filter: {len(dvf):,} rows')

# French decimal format → float
dvf['valeur'] = pd.to_numeric(
    dvf['Valeur fonciere'].str.replace(',', '.', regex=False), errors='coerce'
)
dvf['surface'] = pd.to_numeric(dvf['Surface reelle bati'], errors='coerce')

dvf = dvf[(dvf['surface'] > 0) & (dvf['valeur'] > 0)].copy()
dvf['prix_m2'] = dvf['valeur'] / dvf['surface']
dvf = dvf[dvf['prix_m2'] < 20_000].copy()

# Reconstruct 5-char code_commune
dvf['dept_str'] = dvf['Code departement'].astype(str).str.strip().str.zfill(2)
dvf['comm_str'] = dvf['Code commune'].astype(str).str.strip().str.zfill(3)
dvf['code_commune'] = dvf['dept_str'] + dvf['comm_str']

# Filter to PLM arrondissement codes only
paris_codes   = [f'75{str(i).zfill(3)}' for i in range(101, 121)]
lyon_codes    = [f'69{str(i).zfill(3)}' for i in range(381, 390)]
mars_codes    = [f'13{str(i).zfill(3)}' for i in range(201, 217)]
plm_codes     = paris_codes + lyon_codes + mars_codes

dvf_plm = dvf[dvf['code_commune'].isin(plm_codes)].copy()
print(f'PLM transactions: {len(dvf_plm):,}')
print(dvf_plm['code_commune'].value_counts().sort_index())

In [ ]:
# Median prix/m² per arrondissement code_commune
prix_arrdt = (
    dvf_plm.groupby('code_commune')
    .agg(
        median_prix_m2=('prix_m2', 'median'),
        n_transactions=('prix_m2', 'count')
    )
    .reset_index()
)

print(f'Arrondissements with prix data: {len(prix_arrdt)}')
print(prix_arrdt.sort_values('median_prix_m2', ascending=False).to_string())

## 5. Join prix m² to election secteurs via mapping dict

In [ ]:
# For each secteur, pool all its arrondissements' transactions and take the median
# (rather than averaging per-arrondissement medians: pooling is more accurate)

prix_arrdt_idx = prix_arrdt.set_index('code_commune')

def prix_for_secteur(code_secteur):
    arrdts = SECTEUR_ARRDTS.get(code_secteur, [])
    # Pool transactions from all arrondissements in this secteur
    subset = dvf_plm[dvf_plm['code_commune'].isin(arrdts)]
    if len(subset) == 0:
        return None, 0
    return subset['prix_m2'].median(), len(subset)

secteur_voix[['median_prix_m2', 'n_transactions']] = [
    prix_for_secteur(code) for code in secteur_voix['Code secteur']
]

print(secteur_voix[['ville', 'libelle_secteur', 'winning_bloc', 'median_prix_m2', 'n_transactions']]
      .sort_values(['ville', 'median_prix_m2'], ascending=[True, False])
      .to_string())

## 6. Sanity check

In [ ]:
# Median prix by city: directional check
# Expected: Paris >> Marseille > Lyon (roughly)
print('Median prix/m² by city:')
print(secteur_voix.groupby('ville')['median_prix_m2'].median().sort_values(ascending=False))

print('\nBloc distribution by city:')
print(secteur_voix.groupby(['ville', 'winning_bloc']).size().unstack(fill_value=0))

# Missing prix data
missing_prix = secteur_voix[secteur_voix['median_prix_m2'].isna()]
if len(missing_prix) > 0:
    print('\nWARNING: secteurs with no prix data:')
    print(missing_prix[['Code secteur', 'libelle_secteur']].to_string())
else:
    print('\nAll secteurs have prix data.')

## 7. Export JSON

In [ ]:
import json, os

os.makedirs('../data/processed', exist_ok=True)

export = secteur_voix[[
    'Code secteur', 'ville', 'libelle_secteur',
    'winning_nuance', 'winning_bloc',
    'abstention_rate',
    'median_prix_m2', 'n_transactions'
]].copy()

# Add arrondissement codes for GeoJSON join in the viz layer
export['arrondissement_codes'] = export['Code secteur'].map(SECTEUR_ARRDTS)

export.columns = [
    'code_secteur', 'ville', 'libelle_secteur',
    'winning_nuance', 'winning_bloc',
    'abstention_rate',
    'median_prix_m2', 'n_transactions',
    'arrondissement_codes'
]

output_path = '../data/processed/plm-secteurs-2025.json'
export.to_json(output_path, orient='records', force_ascii=False, indent=2)

print(f'Exported {len(export)} secteurs → {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')
print(export[['code_secteur', 'ville', 'winning_bloc', 'abstention_rate', 'median_prix_m2']].to_string())